
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 랩 - 문서 파싱, 변환 및 청크

## 개요

이 실험실에서는 지정된 볼륨에 저장된 문서 세트를 다루게 됩니다. Python와 Databricks 도구를 사용해 이 문서들을 **파싱**, **변환**, **청크**하는 방법을 배우게 됩니다. 최종 결과는 추가 분석을 위해 Delta 표에 저장됩니다.

## 학습 목표
이 실험실이 끝날 때쯤이면 다음과 같은 일을 할 수 있게 될 것입니다:
1. 문서를 Python으로 **파싱**하세요.
2. JSON 형식에서 파싱된 문서를 **평탄**화하세요.
3. **AI Query 사용**하여 JSON를 Markdown로 변환하세요.
4. Markdown를 일정한 크기로 **분할**하세요.
5. 결과를 Delta 테이블에 **저장**하세요.

## 요구 사항
- 샘플 문서가 포함된 볼륨. 이것은 설정 코드로 만들어졌습니다. 이 작업은 workspace 설정에서 이루어집니다.
- **서버리스 Compute (환경 버전 5)**.
- 서버리스 compute 구성의 **의존성**에 필요한 라이브러리가 추가됩니다.

**📌 당신의 태스크: 이 실험실에서 당신의 태스크는 섹션을 적절한 코드로 교체 `<FILL_IN>` 하는 것입니다.**

## 준비

아래 코드를 실행하여 필요한 라이브러리를 설치하고 교실 환경을 구성하세요.

이 단계는 모든 의존성이 사용 가능하고 워크스페이스가 데모 준비가 완료되도록 보장합니다.

In [0]:
%run ../Includes/Classroom-Setup-02

## 태스크 1: Python을 사용하여 문서 구문 분석

이 섹션에서는 지정된 볼륨에서 문서 집합을 **로드하고 파싱**할 것입니다. Python을 사용하여 파일을 읽고 내용을 파싱하여 추가 처리를 하세요.

**단계:**
1. 제공된 변수를 볼륨 경로로 사용하세요 (예: `docs_path` ).
2. 각 문서를 **`ai_parse_document` AI 함수**로 파싱합니다.
3. 파싱된 결과를 `df_raw`라는 DataFrame에 저장하세요.

아래 코드를 작성하여 이 작업을 수행하세요.

In [0]:
## 지정된 권의 모든 문서를 ai_parse_document로 파싱합니다.
## 파싱된 결과를 df_raw라는 DataFrame에 저장합니다.

from pyspark.sql.functions import expr

## 문서 볼륨의 모든 파일을 바이너리로 읽으세요.
files_df = <FILL_IN>

## 각 문서를 ai_parse_document (버전 2.0)을 사용하여 파싱합니다.
df_raw = files_df.<FILL_IN>

## 더 쉽게 표시하려면 이진 콘텐츠 열을 삭제하세요.
result_df = df_raw.drop("content")
display(result_df)

In [0]:
%skip
## 지정된 권의 모든 문서를 ai_parse_document을 사용하여 파싱합니다.
## 파싱된 결과를 df_raw라는 이름의 DataFrame에 저장하세요.
from pyspark.sql.functions import expr

## 문서 볼륨의 모든 파일을 바이너리로 읽으세요
files_df = spark.read.format("binaryFile").load(user_docs_path)

## 각 문서를 ai_parse_document를 사용하여 파싱하기 (버전 2.0)
df_raw = files_df.withColumn(
   "parsed_content",
   expr(f"ai_parse_document(content, map('version', '2.0', 'imageOutputPath', '{user_docs_path}/parsed_images/'))")
)

## 더 쉽게 표시하려면 이진 내용 열을 빼세요
result_df = df_raw.drop("content")
display(result_df)

## 태스크 2: 파싱된 JSON 문서 평탄화

이 섹션에서는 분석을 용이하게 하고 후속 처리를 위해 **구문 분석된 JSON 콘텐츠를 평면적인 표 형식 구조로 변환**합니다.

**단계:**
1. `parsed_content` 열의 `df_raw`에서 관련 필드를 추출하세요.
2. `df_flat`라는 이름의 새로운 DataFrame을 만드세요.
3. 키 메타데이터와 페이지 수준의 정보를 추출하는 데 집중하세요.

아래 코드를 작성하여 이 작업을 수행하세요.

In [0]:
## 열을 parsed_content 평탄화합니다 df_raw
## 에 '원소' 필드만 추출하세요df_flat

from pyspark.sql.functions import expr

df_flat = df_raw.<FILL_IN>
display(df_flat)

In [0]:
%skip
## 열을 parsed_content 평탄화하여 df_raw
## df_flat에 '원소' 필드만 추출하세요
from pyspark.sql.functions import expr

df_flat = df_raw.select(
   "path",
   expr("parsed_content:document:elements").alias("elements")
)
display(df_flat)

## 태스크 3: JSON를 마크다운으로 변환하기 위해 AI 쿼리 사용

이 섹션에서는 **ai_query** 기능을 사용해 JSON 요소를 깔끔하고 읽기 쉬운 마크다운 형식으로 변환합니다. 이 접근법은 헤더, 테이블, 구조와 같은 문서 의미론을 보존하기 위해 대규모 언어 모델을 활용하여, 출력 결과를 하위 LLM 작업에 더 유용하게 만듭니다.

**단계:**
1. 프롬프트를 사용해 LLM에게 JSON를 마크다운으로 변환하도록 지시하세요.
2. 마크다운 결과를 DataFrame `markdown`에 새 `df_markdown` 열로 저장하세요.

아래 코드를 작성하여 이 작업을 수행하세요.

In [0]:
## JSON 'elements'를 Markdown로 변환해 AI Query

from pyspark.sql.functions import expr, concat, lit, col

## Databricks 기초 모델 엔드포인트를 선택하세요
ENDPOINT = <FILL_IN>

## LLM 프롬프트
prompt_prefix = <FILL_IN>

## Markdown로 요소를 변환하려면 ai_query을 신청하세요
## 프롬프트와 요소를 문자열로 연결하세요
## 텍스트 출력에 responseFormat 지정하기
df_markdown = df_flat.<FILL_IN>
display(df_markdown)

In [0]:
%skip
# JSON '요소'를 Markdown로 변환해 AI Query
from pyspark.sql.functions import expr, concat, lit, col

# Databricks 기초 모델 엔드포인트를 선택하십시오
ENDPOINT = "databricks-claude-sonnet-4-6"

# LLM 프롬프트
prompt_prefix = '''
You are a helpful assistant. Given a JSON object representing document elements, convert the content into clean, readable markdown. Preserve important structure such as headers, tables, and captions. Do not include any JSON or code blocks in the output—just the clean markdown text.

JSON:
'''

# Markdown으로 변환하려면 ai_query을 적용하십시오
# 프롬프트와 요소를 문자열로 연결해
# 텍스트 출력에 responseFormat 지정하기
df_markdown = df_flat.withColumn(
    "markdown",
    expr(f"ai_query('{ENDPOINT}', CONCAT('{prompt_prefix}', CAST(elements AS STRING)))")
)
display(df_markdown)

## 태스크 4: 마크다운을 일정 크기로 청크하기

이 섹션에서는 **마크다운 텍스트를 고정된 크기의 청크로 나누어** 효율적인 검색과 후속 처리를 돕습니다. 청킹을 수행할 때는 langchain-text-splitters 라이브러리를 사용할 것입니다.

**단계:**
1. 일정한 청크 크기(예: 1000자)와 겹침(예: 200자)을 설정하세요.
2. 청크된 결과를 새 DataFrame 이름 `df_chunks`으로 저장하세요.

아래 코드를 작성하여 이 작업을 수행하세요.

In [0]:
## langchain-text-splitters를 사용해 Markdown 텍스트를 일정한 크기로 청크합니다
from pyspark.sql.functions import UDF, col, explode
from pyspark.sql.types import ArrayType, StringType
from langchain_text_splitters import RecursiveCharacterTextSplitter

## parameter
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

splitter = <FILL_IN>

@udf(ArrayType(StringType()))
def split_md(s: str):
    if not s or not s.strip():
        return []
    return [c for c in splitter.split_text(s) if c and c.strip()]

## 분할 마크다운을 조각으로 분해하세요
df_chunks = df_markdown.<FILL_IN>

display(df_chunks)

In [0]:
%skip
# langchain-text-splitters를 사용해 마크다운 텍스트를 일정한 크기로 청크합니다
from pyspark.sql.functions import udf, col, explode
from pyspark.sql.types import ArrayType, StringType
from langchain_text_splitters import RecursiveCharacterTextSplitter


# parameter
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200


splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)


@udf(ArrayType(StringType()))
def split_md(s: str):
    if not s or not s.strip():
        return []
    return [c for c in splitter.split_text(s) if c and c.strip()]


df_chunks = df_markdown.select("path", explode(split_md("markdown")).alias("chunk"))
display(df_chunks)

## 태스크 5: 결과를 Delta 테이블에 저장하기

이 섹션에서는 **청크된 마크다운 결과**를 Delta 테이블에 저장하여 하위 분석 및 Workflows 검색을 위해 사용합니다.

**단계:**
1. 제공된 카탈로그와 스키마 변수를 사용하여 출력 테이블 이름을 정의하세요.
2. 덮어쓰기 모드를 사용해 DataFrame `df_chunks` 를 Delta 테이블에 쓰세요.

아래 코드를 작성하여 이 작업을 수행하세요.

In [0]:
## 청크된 결과를 Delta 테이블에 저장하여 후속 분석을 위해 준비하세요.

## 카탈로그 및 스키마 변수를 사용하여 출력 테이블 이름을 정의합니다
output_table = f"{catalog}.{schema}.lab_chunked_docs"

## DataFrame를 Delta 테이블에 쓰세요
## 그것이 이미 존재한다면 테이블을 덮어쓰기
df_chunks.<FILL_IN>

print(f"✅ Chunked results saved to Delta table: {output_table}")

In [0]:
%skip
# 청크된 결과를 Delta 테이블에 저장하여 후속 분석을 위해
# 카탈로그 및 스키마 변수를 사용하여 출력 테이블 이름을 정의합니다
output_table = f"{catalog}.{schema}.lab_chunked_docs"

# DataFrame을 Delta 테이블에 쓰기
# 이미 존재한다면 테이블 덮어쓰기
df_chunks.write.format("delta").mode("overwrite").saveAsTable(output_table)

print(f" ✅ Delta 테이블에 저장된 청크된 결과: {output_table}")

## 요약과 다음 단계

Python와 Databricks 도구를 사용해 문서 파싱, 변환, 청크 처리를 수행하는 랩을 완료하셨습니다. 다음을 학습하셨습니다:

* 볼륨에서 문서를 파싱하고 구조화된 콘텐츠를 **추출**합니다.
* 파싱된 JSON을 **평면화**하여 관련 요소를 선택합니다.
* **AI 쿼리를 사용**하여 JSON 요소를 마크다운으로 변환합니다.
* 효율적인 검색을 위해 마크다운 텍스트를 일정 크기로 **분할**합니다.
* 최종 결과를 델타 테이블에 저장하여 후속 분석을 **수행**합니다.

**다음 단계 (선택 사항):**
* AI Search 또는 LLM 기반 검색을 이용해 청크 데이터를 삽입하고 검색하는 방법을 탐험합니다.
* 다양한 청크 크기와 프롬프트를 사용해 워크플로를 최적화하세요.
* Delta 표를 검토하고 사용 사례에 맞는 결과를 검증하세요.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>